# Clase 11 — Correlación y Análisis Categórico

**Diplomado en Data Science Aplicada con Python para la Toma de Decisiones**  
Arca Continental Ecuador | UDLA

---

## Objetivos de hoy

1. **Correlación**: medir la relación entre dos variables (Pearson y Spearman)
2. **Visualizar relaciones**: heatmaps, scatter plots, regplot
3. **Análisis categórico**: crear grupos, comparar, tablas cruzadas
4. **Pregunta de negocio**: ¿qué hace que un vino sea bueno?

**Dataset**: Wine Quality (el mismo de la clase 10).

## 0. Imports y carga de datos

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

url = ("https://archive.ics.uci.edu/ml/"
       "machine-learning-databases/wine-quality/"
       "winequality-red.csv")
df = pd.read_csv(url, sep=";")
print(f"{df.shape[0]} vinos, {df.shape[1]} variables")
df.head()

---

# PARTE 1: Comparar dos variables — el scatter plot

Hasta ahora analizamos variables **una a la vez** (histograma, boxplot, estadísticas). Ahora queremos responder: **¿hay relación entre dos variables?**

La herramienta visual para esto es el **scatter plot** (gráfico de dispersión):
- Cada **punto** es una observación (un vino)
- **Eje X**: una variable (ej: alcohol)
- **Eje Y**: otra variable (ej: quality)
- Buscamos **patrones** en la nube de puntos

## 1.1 Nuestro primer scatter plot

¿Los vinos con más alcohol tienen mejor nota?

In [ ]:
# Cada punto es un vino
# x = alcohol, y = quality
sns.scatterplot(x="alcohol", y="quality", data=df, alpha=0.3)
plt.xlabel("Alcohol (% vol)")
plt.ylabel("Quality (nota 0-10)")
plt.title("Alcohol vs Quality")
plt.show()

In [ ]:
# Relación negativa: más ácido acético → peor nota
sns.scatterplot(x="volatile acidity", y="quality", data=df, alpha=0.3, color="#C82B40")
plt.xlabel("Acidez volátil")
plt.ylabel("Quality (nota 0-10)")
plt.title("Acidez volátil vs Quality")
plt.show()

In [ ]:
# Sin relación clara: el azúcar no parece afectar la calidad
sns.scatterplot(x="residual sugar", y="quality", data=df, alpha=0.3, color="#6B7280")
plt.xlabel("Azúcar residual (g/L)")
plt.ylabel("Quality (nota 0-10)")
plt.title("Azúcar residual vs Quality")
plt.show()

## 1.2 Del scatter plot a un número: la correlación

El scatter plot nos muestra el patrón **visualmente**. Pero queremos un **número** que resuma esa relación:

- **r ≈ +1**: los puntos "suben" → relación positiva (alcohol vs quality)
- **r ≈ 0**: los puntos no tienen patrón → sin relación lineal (azúcar vs quality)
- **r ≈ -1**: los puntos "bajan" → relación negativa (acidez vs quality)

> **Importante: r ≈ 0 NO significa "no hay relación". Significa que no hay relación LINEAL.**

## 1.3 Heatmap de correlación

`df.corr()` calcula la correlación entre **todas** las columnas de una vez.

In [ ]:
# Heatmap de Pearson
corr = df.corr(numeric_only=True)

sns.heatmap(corr, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5)
plt.title("Correlación Pearson — Wine Quality")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap de Spearman (usa rangos, robusto a outliers)
corr_sp = df.corr(method="spearman", numeric_only=True)

sns.heatmap(corr_sp, annot=True, fmt=".2f",
            cmap="RdBu_r", center=0, vmin=-1, vmax=1,
            linewidths=0.5)
plt.title("Correlación Spearman — Wine Quality")
plt.tight_layout()
plt.show()

In [ ]:
# ¿Qué variables correlacionan más con quality?
corr_quality = corr["quality"].drop("quality").sort_values(ascending=False)
print("Correlación con quality (Pearson):")
print(corr_quality.round(3))

## 1.3 Correlación ≠ Causalidad

| | |
|---|---|
| **Mito** | "El alcohol causa que el vino sea bueno" (porque r ≈ 0.48) |
| **Realidad** | Los vinos con más alcohol suelen ser de uvas más maduras, que producen mejor sabor. La **madurez de la uva** es la variable oculta. |
| **Regla** | Correlación implica que **vale la pena investigar** si hay causalidad. No la prueba. |

> Ejemplo clásico: el consumo de helado y los ahogamientos correlacionan positivamente. ¿El helado causa ahogamientos? No — ambos aumentan en verano (variable oculta: temperatura).

---

# PARTE 2: Pearson vs Spearman

| | Pearson (r) | Spearman (ρ) |
|---|---|---|
| **Mide** | Relación **lineal** | Relación **monótona** (no necesita ser recta) |
| **Usa** | Los **valores** originales | Los **rangos** (posiciones al ordenar) |
| **Outliers** | Sensible (un extremo lo distorsiona) | Robusto (el rango no cambia mucho) |
| **Cuándo** | Datos normales, sin outliers | Datos sesgados, outliers, relación curva |

In [ ]:
# Comparar ambos métodos
cols = ["alcohol", "volatile acidity", "citric acid",
        "sulphates", "residual sugar", "chlorides"]

comparison = pd.DataFrame({
    "Pearson": [df[c].corr(df["quality"]) for c in cols],
    "Spearman": [df[c].corr(df["quality"], method="spearman") for c in cols],
}, index=cols)
comparison["Diferencia"] = abs(comparison["Pearson"] - comparison["Spearman"])
comparison = comparison.sort_values("Diferencia", ascending=False)

print("Pearson vs Spearman (correlación con quality):")
print(comparison.round(3))
print("\n→ Donde la diferencia es grande, la relación probablemente")
print("  NO es lineal o hay outliers influyentes.")

## 2.1 regplot: scatter plot + línea de ajuste

`sns.regplot()` dibuja los puntos y ajusta una línea. Tiene dos modos:
- **Lineal** (por defecto): busca la recta que mejor se ajusta (mínimos cuadrados)
- **LOWESS** (`lowess=True`): ajusta una curva suave usando vecinos cercanos

In [ ]:
# Relación lineal: alcohol vs quality
sns.regplot(x="alcohol", y="quality", data=df,
            scatter_kws={"alpha": 0.15},
            line_kws={"color": "#C82B40"})
plt.title("alcohol vs quality (lineal)")
plt.show()

In [ ]:
# Relación no lineal: citric acid vs volatile acidity
sns.regplot(x="citric acid", y="volatile acidity", data=df,
            scatter_kws={"alpha": 0.15},
            line_kws={"color": "#C82B40"},
            lowess=True)
plt.title("citric acid vs volatile acidity (curva suave - lowess)")
plt.show()

### Ejercicio 1: Correlación

1. Compara los heatmaps de Pearson y Spearman que acabamos de hacer: ¿qué diferencias ves?
2. Elige las 2 variables más correlacionadas con `quality` y haz `regplot` de cada una
3. Prueba `lowess=True` en una de ellas: ¿la curva es diferente a la recta?

In [ ]:
# Tu código aquí


---

# PARTE 3: Análisis Categórico

Hasta ahora analizamos variables **una a la vez** o **pares de continuas** (correlación).

Ahora vamos a responder la pregunta de negocio: **¿qué diferencia a un vino bueno de uno malo?**

Para eso necesitamos **comparar grupos**.

## 3.1 Crear categorías con `pd.cut`

`pd.cut` divide una variable numérica en intervalos que tú defines.

In [ ]:
df["quality_group"] = pd.cut(
    df["quality"],
    bins=[0, 4, 6, 10],
    labels=["Bajo (3-4)", "Medio (5-6)", "Alto (7-8)"]
)

print(df["quality_group"].value_counts().sort_index())

In [ ]:
# Distribución visual
df["quality_group"].value_counts().sort_index().plot(
    kind="bar", color=["#C82B40", "#6B7280", "#16A34A"],
    edgecolor="white")
plt.title("Distribución de calidad")
plt.ylabel("Cantidad de vinos")
plt.xticks(rotation=0)
plt.show()

## 3.2 Comparar grupos con `groupby`

In [ ]:
cols = ["alcohol", "volatile acidity", "sulphates", "citric acid"]

summary = df.groupby("quality_group")[cols].agg(["mean", "median"])
print(summary.round(3))

In [ ]:
# Mediana por grupo (resumen más compacto)
perfil = df.groupby("quality_group")[cols].median()
print(perfil.round(3))

## 3.3 Boxplot por grupo

La forma más directa de comparar distribuciones entre categorías.

In [ ]:
sns.boxplot(x="quality_group", y="alcohol", data=df)
plt.title("Alcohol por grupo de calidad")
plt.show()

In [ ]:
sns.boxplot(x="quality_group", y="volatile acidity", data=df)
plt.title("Acidez volátil por grupo de calidad")
plt.show()

In [ ]:
sns.boxplot(x="quality_group", y="sulphates", data=df)
plt.title("Sulphates por grupo de calidad")
plt.show()

In [ ]:
sns.boxplot(x="quality_group", y="citric acid", data=df)
plt.title("Ácido cítrico por grupo de calidad")
plt.show()

## 3.4 Violinplot: ver la forma de la distribución

El violinplot es un **boxplot + histograma** combinados:

- **El ancho** del violín muestra dónde se concentran los datos (como un histograma girado y espejado)
- **Las 3 líneas internas** (con `inner="quartile"`) son Q1, mediana y Q3 (la misma info del boxplot)
- **Ventaja sobre boxplot**: ves si la distribución es simétrica, sesgada, o tiene formas inesperadas

In [ ]:
sns.violinplot(x="quality_group", y="alcohol", data=df, inner="quartile")
plt.title("Alcohol por grupo de calidad")
plt.show()

In [ ]:
sns.violinplot(x="quality_group", y="volatile acidity", data=df, inner="quartile")
plt.title("Acidez volátil por grupo de calidad")
plt.show()

## 3.5 Countplot: frecuencias de categorías

In [ ]:
sns.countplot(x="quality", data=df, color="#C82B40")
plt.title("Distribución de quality (notas individuales)")
plt.show()

## 3.6 Tablas de contingencia con `pd.crosstab`

`pd.crosstab` cuenta cuántos casos hay en cada combinación de dos variables categóricas.

In [ ]:
# Crear grupos de alcohol
df["alcohol_group"] = pd.cut(
    df["alcohol"], bins=3,
    labels=["Bajo", "Medio", "Alto"]
)

# Tabla cruzada: conteos
tabla = pd.crosstab(df["quality_group"], df["alcohol_group"], margins=True)
print(tabla)

In [ ]:
# Tabla cruzada: porcentajes por fila
tabla_pct = pd.crosstab(
    df["quality_group"], df["alcohol_group"],
    normalize="index"
)
print(tabla_pct.round(3))

In [ ]:
# Visualizar la tabla como heatmap
sns.heatmap(tabla_pct, annot=True, fmt=".1%",
            cmap="YlOrRd", linewidths=0.5)
plt.title("% de nivel de alcohol por grupo de calidad")
plt.show()

### Ejercicio 2: Análisis por grupos

1. Crea una variable `acidity_group` usando `pd.cut` en `volatile acidity` con `bins=3` y `labels=["Baja", "Media", "Alta"]`
2. Haz un `crosstab` de `quality_group` vs `acidity_group` (con `normalize="index"`)
3. Haz un boxplot de `alcohol` agrupado por `acidity_group`
4. **Pregunta**: ¿los vinos con acidez alta tienden a ser buenos o malos?

In [ ]:
# Tu código aquí


---

# PARTE 4: Conectando todo — ¿Qué hace un buen vino?

## 4.1 Perfil de cada grupo

In [ ]:
cols = ["alcohol", "volatile acidity", "citric acid",
        "sulphates", "chlorides", "residual sugar"]

perfil = df.groupby("quality_group")[cols].median().T
print("Mediana de cada variable por grupo de calidad:")
print(perfil.round(3))

## 4.2 Los outliers de la clase pasada: ¿buenos o malos?

In [ ]:
col = "chlorides"
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1

outliers = df[(df[col] < Q1 - 1.5 * IQR) | (df[col] > Q3 + 1.5 * IQR)]
normales = df[~df.index.isin(outliers.index)]

print(f"Outliers: {len(outliers)} vinos")
print(f"  Quality promedio: {outliers['quality'].mean():.2f}")
print(f"\nNormales: {len(normales)} vinos")
print(f"  Quality promedio: {normales['quality'].mean():.2f}")

In [ ]:
# ¿En qué grupos de calidad caen los outliers?
print("Distribución de los outliers de chlorides:")
print(outliers["quality_group"].value_counts().sort_index())
print("\nDistribución de los normales:")
print(normales["quality_group"].value_counts(normalize=True).sort_index().round(3))

## 4.3 ¿Afectan los outliers a la correlación?

In [ ]:
col = "chlorides"
Q1 = df[col].quantile(0.25)
Q3 = df[col].quantile(0.75)
IQR = Q3 - Q1
df_clean = df[(df[col] >= Q1 - 1.5 * IQR) & (df[col] <= Q3 + 1.5 * IQR)]

r_con = df[col].corr(df["quality"])
r_sin = df_clean[col].corr(df_clean["quality"])

rho_con = df[col].corr(df["quality"], method="spearman")
rho_sin = df_clean[col].corr(df_clean["quality"], method="spearman")

print(f"Con outliers:  Pearson={r_con:.3f},  Spearman={rho_con:.3f}")
print(f"Sin outliers:  Pearson={r_sin:.3f},  Spearman={rho_sin:.3f}")

---

# Ejercicio Integrador

Elige **una** variable: `sulphates` o `total sulfur dioxide` y haz el análisis completo.

### Parte A: Distribución

1. Haz un histograma. ¿Es normal? Verifica con Shapiro-Wilk.
2. Detecta outliers con IQR.

In [ ]:
# Tu código aquí


### Parte B: Correlación

3. Calcula Pearson y Spearman con `quality`.
4. Haz un `regplot` (lineal y lowess).

In [ ]:
# Tu código aquí


### Parte C: Grupos

5. Compara la mediana de tu variable por `quality_group`.
6. Haz un boxplot por `quality_group`.
7. **Pregunta**: ¿el patrón que ves confirma lo que decía la correlación?

In [ ]:
# Tu código aquí
